In [1]:
# ============================================================
# CELL 1: Mount Drive & Cài thư viện
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

!pip install -q datasets transformers torchvision scikit-learn tqdm Pillow

Mounted at /content/drive


In [2]:
# ============================================================
# CELL 2: Import & Config
# ============================================================
import os, json, warnings
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from datasets import load_dataset
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm import tqdm
from datetime import datetime

warnings.filterwarnings("ignore")

DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR   = "/content/drive/MyDrive/ResNet_MiRAGe"
BATCH_SIZE = 16
EPOCHS     = 5
LR         = 2e-4
MAX_LEN    = 256

os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Device  : {DEVICE}")
print(f"Save dir: {SAVE_DIR}")

Device  : cuda
Save dir: /content/drive/MyDrive/ResNet_MiRAGe


In [3]:
# ============================================================
# CELL 3: Load dataset
# ============================================================
print("Đang tải dataset anson-huang/mirage-news ...")
raw = load_dataset("anson-huang/mirage-news")
print(raw)
print("\nFeatures:", raw["train"].features)
print("Mẫu đầu :", {k: str(v)[:80] for k, v in raw["train"][0].items()})

test_splits = [k for k in raw.keys() if k not in ("train", "validation")]
print(f"\nTest splits: {test_splits}")

Đang tải dataset anson-huang/mirage-news ...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/655M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/143M [00:00<?, ?B/s]

data/test1_nyt_mj-00000-of-00001.parquet:   0%|          | 0.00/20.2M [00:00<?, ?B/s]

...

Generating test2_bbc_dalle split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test3_cnn_dalle split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test4_bbc_sdxl split:   0%|          | 0/500 [00:00<?, ? examples/s]

Generating test5_cnn_sdxl split:   0%|          | 0/500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 10000
    })
    validation: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 2500
    })
    test1_nyt_mj: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test2_bbc_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test3_cnn_dalle: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test4_bbc_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
    test5_cnn_sdxl: Dataset({
        features: ['image', 'label', 'text'],
        num_rows: 500
    })
})

Features: {'image': Image(mode=None, decode=True), 'label': ClassLabel(names=['real', 'fake']), 'text': Value('string')}
Mẫu đầu : {'image': '<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=600x353 at 0x7CF1ED4AA5A0', 'label': '1', 'text': 'Andal Amp

In [4]:
# ============================================================
# CELL 4: Dataset class (image-only ResNet)
# ============================================================
IMG_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

IMG_TRANSFORM_TRAIN = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

class MiRAGeResNetDataset(Dataset):
    def __init__(self, hf_dataset, is_train=False):
        self.data     = hf_dataset
        self.transform = IMG_TRANSFORM_TRAIN if is_train else IMG_TRANSFORM

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # IMAGE
        try:
            img = item.get("image") or item.get("img")
            if img is None:
                raise ValueError("No image")
            img = img.convert("RGB") if isinstance(img, Image.Image) \
                  else Image.fromarray(np.array(img)).convert("RGB")
            image_tensor = self.transform(img)
        except Exception:
            image_tensor = torch.zeros(3, 224, 224)

        # LABEL
        label = int(item.get("label", item.get("labels", 0)))

        return image_tensor, label

In [5]:
# ============================================================
# CELL 5: Model ResNet50 + Classifier
# ============================================================
class ResNetFakeNewsClassifier(nn.Module):
    def __init__(self, num_classes=2, dropout=0.4):
        super().__init__()

        # Backbone ResNet50
        backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        # Bỏ FC layer cuối, lấy feature extractor
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])

        # Classifier mới
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, image_tensor):
        feat = self.backbone(image_tensor)   # (B, 2048, 1, 1)
        return self.classifier(feat)

In [6]:
# ============================================================
# CELL 6: Khởi tạo dataloader
# ============================================================
train_ds = MiRAGeResNetDataset(raw["train"],      is_train=True)
val_ds   = MiRAGeResNetDataset(raw["validation"], is_train=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"Train : {len(train_ds):,} mẫu | {len(train_loader):,} steps/epoch")
print(f"Val   : {len(val_ds):,} mẫu   | {len(val_loader):,} steps/epoch")
for s in test_splits:
    print(f"Test [{s}]: {len(raw[s]):,} mẫu")

Train : 10,000 mẫu | 625 steps/epoch
Val   : 2,500 mẫu   | 157 steps/epoch
Test [test1_nyt_mj]: 500 mẫu
Test [test2_bbc_dalle]: 500 mẫu
Test [test3_cnn_dalle]: 500 mẫu
Test [test4_bbc_sdxl]: 500 mẫu
Test [test5_cnn_sdxl]: 500 mẫu


In [7]:
# ============================================================
# CELL 7: Training + lưu latest checkpoint
# ============================================================
model     = ResNetFakeNewsClassifier().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)

LATEST_CKPT = os.path.join(SAVE_DIR, "latest_epoch.pth")

# Resume nếu có checkpoint
start_epoch = 1
history     = []
if os.path.exists(LATEST_CKPT):
    print("Tìm thấy checkpoint, đang resume...")
    ckpt        = torch.load(LATEST_CKPT, map_location=DEVICE)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt["epoch"] + 1
    history     = ckpt.get("history", [])
    print(f"Tiếp tục từ epoch {start_epoch} | "
          f"Val Acc: {ckpt['val_acc']:.4f} | Val F1: {ckpt['val_f1']:.4f}")

# Hàm evaluate
def evaluate(loader, split_name="Val"):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0
    with torch.no_grad():
        for imgs, labels in tqdm(loader,
                                 desc=f"  Eval [{split_name}]",
                                 leave=False):
            logits     = model(imgs.to(DEVICE))
            loss       = criterion(logits, labels.to(DEVICE))
            total_loss += loss.item()
            all_preds.extend(torch.argmax(logits, 1).cpu().tolist())
            all_labels.extend(labels.tolist())

    acc = accuracy_score(all_labels, all_preds)
    f1  = f1_score(all_labels, all_preds, average="macro")
    print(f"  [{split_name}] Loss: {total_loss/len(loader):.4f} "
          f"| Acc: {acc:.4f} | F1: {f1:.4f}")
    return total_loss / len(loader), acc, f1, all_preds, all_labels

# Training loop
for epoch in range(start_epoch, EPOCHS + 1):
    model.train()
    running_loss              = 0
    train_preds, train_labels = [], []

    loop = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]")
    for imgs, labels in loop:
        optimizer.zero_grad()
        logits = model(imgs.to(DEVICE))
        loss   = criterion(logits, labels.to(DEVICE))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running_loss += loss.item()
        train_preds.extend(torch.argmax(logits, 1).cpu().tolist())
        train_labels.extend(labels.tolist())
        loop.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / len(train_loader)
    train_acc  = accuracy_score(train_labels, train_preds)
    train_f1   = f1_score(train_labels, train_preds, average="macro")
    print(f"\nEpoch {epoch} Train → "
          f"Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")

    val_loss, val_acc, val_f1, _, _ = evaluate(val_loader, "Val")
    scheduler.step()

    history.append({
        "epoch":      epoch,
        "train_loss": round(train_loss, 4),
        "train_acc":  round(train_acc,  4),
        "train_f1":   round(train_f1,   4),
        "val_loss":   round(val_loss,   4),
        "val_acc":    round(val_acc,    4),
        "val_f1":     round(val_f1,     4),
    })

    # Lưu latest checkpoint (ghi đè sau mỗi epoch)
    torch.save({
        "epoch":           epoch,
        "model_state":     model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "val_acc":         val_acc,
        "val_f1":          val_f1,
        "history":         history,
    }, LATEST_CKPT)
    print(f"  ✅ latest_epoch.pth đã lưu (epoch {epoch})")

print("\n=== TRAINING HOÀN TẤT ===")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 201MB/s]
Epoch 1/5 [Train]: 100%|██████████| 625/625 [01:57<00:00,  5.30it/s, loss=0.0536]



Epoch 1 Train → Loss: 0.3394 | Acc: 0.8630 | F1: 0.8630


  [Val] Loss: 0.2706 | Acc: 0.9044 | F1: 0.9044
  ✅ latest_epoch.pth đã lưu (epoch 1)


Epoch 2/5 [Train]: 100%|██████████| 625/625 [01:58<00:00,  5.25it/s, loss=0.2902]


...


Epoch 4 Train → Loss: 0.1481 | Acc: 0.9555 | F1: 0.9555


  [Val] Loss: 0.2018 | Acc: 0.9384 | F1: 0.9383
  ✅ latest_epoch.pth đã lưu (epoch 4)


Epoch 5/5 [Train]: 100%|██████████| 625/625 [02:02<00:00,  5.09it/s, loss=0.0423]



Epoch 5 Train → Loss: 0.0937 | Acc: 0.9719 | F1: 0.9719


  [Val] Loss: 0.0933 | Acc: 0.9744 | F1: 0.9744
  ✅ latest_epoch.pth đã lưu (epoch 5)

=== TRAINING HOÀN TẤT ===


In [8]:
# ============================================================
# CELL 8: Đánh giá tất cả 5 test splits
# ============================================================
ckpt = torch.load(LATEST_CKPT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"✅ Load epoch {ckpt['epoch']} | "
      f"Val Acc: {ckpt['val_acc']:.4f} | Val F1: {ckpt['val_f1']:.4f}\n")

timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
all_results = {}

for split in test_splits:
    print(f"{'='*55}")
    print(f"Split: {split}  ({len(raw[split]):,} mẫu)")
    print('='*55)

    dl = DataLoader(
        MiRAGeResNetDataset(raw[split], is_train=False),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=2
    )

    preds_all, labels_all = [], []
    with torch.no_grad():
        for imgs, labels in tqdm(dl, desc=split):
            logits = model(imgs.to(DEVICE))
            preds_all.extend(torch.argmax(logits, 1).cpu().tolist())
            labels_all.extend(labels.tolist())

    acc    = accuracy_score(labels_all, preds_all)
    f1     = f1_score(labels_all, preds_all, average="macro")
    f1_w   = f1_score(labels_all, preds_all, average="weighted")
    report = classification_report(labels_all, preds_all,
                                   target_names=["Real", "AI-Generated"])
    print(report)
    print(f"Accuracy: {acc:.4f} | F1 Macro: {f1:.4f} | F1 Weighted: {f1_w:.4f}\n")

    all_results[split] = {
        "accuracy":    float(acc),
        "f1_macro":    float(f1),
        "f1_weighted": float(f1_w),
        "report":      classification_report(
                           labels_all, preds_all,
                           target_names=["Real", "AI-Generated"],
                           output_dict=True)
    }

# Bảng tóm tắt
print(f"\n{'='*67}")
print("TỔNG KẾT TẤT CẢ TEST SPLITS")
print(f"{'='*67}")
print(f"{'Split':<35} | {'Accuracy':>8} | {'F1 Macro':>8} | {'F1 Weighted':>11}")
print("-"*67)
for split, res in all_results.items():
    print(f"{split:<35} | {res['accuracy']:>8.4f} | "
          f"{res['f1_macro']:>8.4f} | {res['f1_weighted']:>11.4f}")

✅ Load epoch 5 | Val Acc: 0.9744 | Val F1: 0.9744

Split: test1_nyt_mj  (500 mẫu)


test1_nyt_mj: 100%|██████████| 32/32 [00:03<00:00, 10.34it/s]


              precision    recall  f1-score   support

        Real       0.99      0.97      0.98       250
AI-Generated       0.97      0.99      0.98       250

    accuracy                           0.98       500
   macro avg       0.98      0.98      0.98       500
weighted avg       0.98      0.98      0.98       500

Accuracy: 0.9800 | F1 Macro: 0.9800 | F1 Weighted: 0.9800

Split: test2_bbc_dalle  (500 mẫu)


test2_bbc_dalle: 100%|██████████| 32/32 [00:10<00:00,  3.18it/s]


              precision    recall  f1-score   support

        Real       0.72      0.85      0.78       250
AI-Generated       0.81      0.67      0.73       250

    accuracy                           0.76       500
   macro avg       0.77      0.76      0.76       500
weighted avg       0.77      0.76      0.76       500

Accuracy: 0.7580 | F1 Macro: 0.7560 | F1 Weighted: 0.7560

Split: test3_cnn_dalle  (500 mẫu)


...

              precision    recall  f1-score   support

        Real       0.68      0.85      0.76       250
AI-Generated       0.80      0.60      0.69       250

    accuracy                           0.73       500
   macro avg       0.74      0.73      0.72       500
weighted avg       0.74      0.73      0.72       500

Accuracy: 0.7260 | F1 Macro: 0.7219 | F1 Weighted: 0.7219

Split: test4_bbc_sdxl  (500 mẫu)


test4_bbc_sdxl: 100%|██████████| 32/32 [00:04<00:00,  6.55it/s]


              precision    recall  f1-score   support

        Real       0.69      0.83      0.75       250
AI-Generated       0.79      0.63      0.70       250

    accuracy                           0.73       500
   macro avg       0.74      0.73      0.73       500
weighted avg       0.74      0.73      0.73       500

Accuracy: 0.7300 | F1 Macro: 0.7272 | F1 Weighted: 0.7272

Split: test5_cnn_sdxl  (500 mẫu)


test5_cnn_sdxl: 100%|██████████| 32/32 [00:06<00:00,  4.94it/s]

              precision    recall  f1-score   support

        Real       0.74      0.84      0.79       250
AI-Generated       0.82      0.70      0.76       250

    accuracy                           0.77       500
   macro avg       0.78      0.77      0.77       500
weighted avg       0.78      0.77      0.77       500

Accuracy: 0.7740 | F1 Macro: 0.7729 | F1 Weighted: 0.7729


TỔNG KẾT TẤT CẢ TEST SPLITS
Split                               | Accuracy | F1 Macro | F1 Weighted
-------------------------------------------------------------------
test1_nyt_mj                        |   0.9800 |   0.9800 |      0.9800
test2_bbc_dalle                     |   0.7580 |   0.7560 |      0.7560
test3_cnn_dalle                     |   0.7260 |   0.7219 |      0.7219
test4_bbc_sdxl                      |   0.7300 |   0.7272 |      0.7272
test5_cnn_sdxl                      |   0.7740 |   0.7729 |      0.7729


In [9]:
# ============================================================
# CELL 9: Lưu tất cả kết quả
# ============================================================
# 1. Weights model (inference)
weights_path = os.path.join(SAVE_DIR, "resnet_weights_final.pth")
torch.save(model.state_dict(), weights_path)
print(f"✅ Weights         : {weights_path}")

# 2. Full checkpoint kèm timestamp
full_ckpt_path = os.path.join(SAVE_DIR, f"resnet_full_{timestamp}.pth")
torch.save({
    "epoch":           ckpt["epoch"],
    "model_state":     model.state_dict(),
    "optimizer_state": ckpt["optimizer_state"],
    "scheduler_state": ckpt["scheduler_state"],
    "val_acc":         ckpt["val_acc"],
    "val_f1":          ckpt["val_f1"],
    "history":         ckpt["history"],
}, full_ckpt_path)
print(f"✅ Full checkpoint : {full_ckpt_path}")

# 3. JSON đầy đủ
json_data = {
    "timestamp":        timestamp,
    "checkpoint_epoch": int(ckpt["epoch"]),
    "val_acc":          float(ckpt["val_acc"]),
    "val_f1":           float(ckpt["val_f1"]),
    "training_history": ckpt["history"],
    "test_results":     all_results,
    "config": {
        "model":         "ResNet50",
        "image_encoder": "resnet50-imagenet1k",
        "dataset":       "anson-huang/mirage-news",
        "epochs":        EPOCHS,
        "batch_size":    BATCH_SIZE,
        "lr":            LR,
    }
}
json_path = os.path.join(SAVE_DIR, f"results_all_{timestamp}.json")
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_data, f, indent=2, ensure_ascii=False)
print(f"✅ JSON report     : {json_path}")

# 4. TXT dễ đọc
txt_path = os.path.join(SAVE_DIR, f"results_all_{timestamp}.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(f"ResNet50 | MiRAGeNews | {timestamp}\n")
    f.write("=" * 67 + "\n")
    f.write(f"Model        : resnet50 (image-only)\n")
    f.write(f"Dataset      : anson-huang/mirage-news\n")
    f.write(f"Epochs       : {EPOCHS} | Batch: {BATCH_SIZE} | LR: {LR}\n")
    f.write(f"Checkpoint   : epoch {ckpt['epoch']}\n")
    f.write(f"Val Acc      : {ckpt['val_acc']:.4f}\n")
    f.write(f"Val F1 Macro : {ckpt['val_f1']:.4f}\n")
    f.write("\n--- Training History ---\n")
    f.write(f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | "
            f"{'Train F1':>8} | {'Val Loss':>8} | {'Val Acc':>7} | {'Val F1':>7}\n")
    f.write("-" * 75 + "\n")
    for h in ckpt["history"]:
        f.write(f"{h['epoch']:>6} | {h['train_loss']:>10.4f} | {h['train_acc']:>9.4f} | "
                f"{h['train_f1']:>8.4f} | {h['val_loss']:>8.4f} | "
                f"{h['val_acc']:>7.4f} | {h['val_f1']:>7.4f}\n")
    f.write(f"\n--- Test Results ---\n")
    f.write(f"{'Split':<35} | {'Accuracy':>8} | {'F1 Macro':>8} | {'F1 Weighted':>11}\n")
    f.write("-" * 67 + "\n")
    for split, res in all_results.items():
        f.write(f"{split:<35} | {res['accuracy']:>8.4f} | "
                f"{res['f1_macro']:>8.4f} | {res['f1_weighted']:>11.4f}\n")
print(f"✅ TXT report      : {txt_path}")

# 5. Liệt kê tất cả file trong Drive
print(f"\n{'='*65}")
print(f"TẤT CẢ FILE ĐÃ LƯU tại: {SAVE_DIR}")
print(f"{'='*65}")
for fname in sorted(os.listdir(SAVE_DIR)):
    fpath = os.path.join(SAVE_DIR, fname)
    size  = os.path.getsize(fpath) / 1e6
    print(f"  {fname:<50} {size:>7.1f} MB")

✅ Weights         : /content/drive/MyDrive/ResNet_MiRAGe/resnet_weights_final.pth
✅ Full checkpoint : /content/drive/MyDrive/ResNet_MiRAGe/resnet_full_20260406_075643.pth
✅ JSON report     : /content/drive/MyDrive/ResNet_MiRAGe/results_all_20260406_075643.json
✅ TXT report      : /content/drive/MyDrive/ResNet_MiRAGe/results_all_20260406_075643.txt

TẤT CẢ FILE ĐÃ LƯU tại: /content/drive/MyDrive/ResNet_MiRAGe
  latest_epoch.pth                                     295.9 MB
  resnet_full_20260406_075643.pth                      296.0 MB
  resnet_weights_final.pth                              98.8 MB
  results_all_20260406_075643.json                       0.0 MB
  results_all_20260406_075643.txt                        0.0 MB
